In [20]:
import math
import torch
from modelscope import MsDataset

ds_train=MsDataset.load('DAMO_NLP/jd',subset_name='default',split='train')

ds_train_torch = ds_train.to_torch_dataset().filter(
    lambda x: isinstance(x['sentence'], str) and x['label'] is not None and not math.isnan(x['label']))

2025-07-29 14:24:19,265 - modelscope - WARNING - Use trust_remote_code=True. Will invoke codes from jd. Please make sure that you can trust the external codes.
2025-07-29 14:24:20,898 - modelscope - WARNING - Reusing dataset dataset_builder (C:\Users\17246\.cache\modelscope\hub\datasets\DAMO_NLP\jd\master\data_files)
2025-07-29 14:24:20,899 - modelscope - INFO - Generating dataset dataset_builder (C:\Users\17246\.cache\modelscope\hub\datasets\DAMO_NLP\jd\master\data_files)
2025-07-29 14:24:20,903 - modelscope - INFO - Reusing cached meta-data file: C:\Users\17246\.cache\modelscope\hub\datasets\DAMO_NLP\jd\master\data_files\3a0b7ca43b11a413d66fb247f31fb16f
Filter: 100%|██████████| 45366/45366 [00:01<00:00, 36070.62 examples/s]


In [21]:
from torch.utils.data import Dataset, DataLoader
from modelscope import AutoTokenizer

ds_train_torch=list(ds_train_torch)[:200]

tokenizer=AutoTokenizer.from_pretrained('tiansz/bert-base-chinese')

class BertDataset(Dataset):
    def __init__(self,dataset):
        self.dataset=dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        item=self.dataset[index]
        text=item['sentence']
        encoding=tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=128,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return encoding['input_ids'][0],encoding['attention_mask'][0],torch.tensor(item['label'])

train_dataset=BertDataset(ds_train_torch)
train_dataloader=DataLoader(train_dataset, batch_size=2, shuffle=True)

for input_ids, attention_mask, labels in train_dataloader:
    print(input_ids.shape)
    print(attention_mask.shape)
    print(labels.shape)
    break


torch.Size([2, 128])
torch.Size([2, 128])
torch.Size([2])


C:\Users\17246\AppData\Local\Temp\ipykernel_22948\1373134724.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return encoding['input_ids'][0],encoding['attention_mask'][0],torch.tensor(item['label'])


In [22]:
from torch import nn
from modelscope import AutoModel

class Bert(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert=AutoModel.from_pretrained('tiansz/bert-base-chinese')
        self.fc=nn.Linear(768,2)
    def forward(self,input_ids,attention_mask):
        outputs=self.bert(input_ids,attention_mask)
        pooled_output=outputs.pooler_output
        logits=self.fc(pooled_output)
        return logits

In [23]:
from torch import optim

model=Bert()
optimizer=optim.Adam(model.parameters(),lr=2e-5)
criterion=nn.CrossEntropyLoss()
for epoch in range(10):
    for i,(input_ids, attention_mask, labels) in enumerate(train_dataloader):
        outputs=model(input_ids,attention_mask)
        loss=criterion(outputs,labels.long())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f'Epoch [{epoch+1}/10], Loss: {loss.item():.4f}')


2025-07-29 14:26:41,843 - modelscope - INFO - Got 5 files, start to download ...
Processing 5 items:   0%|          | 0.00/5.00 [00:00<?, ?it/s]














Processing 5 items:  20%|██        | 1.00/5.00 [00:00<00:01, 2.16it/s]





Processing 5 items:  60%|██████    | 3.00/5.00 [00:00<00:00, 6.10it/s]




































































































































































































Processing 5 items: 100%|██████████| 5.00/5.00 [00:12<00:00, 2.48s/it]
2025-07-29 14:26:54,246 - modelscope - INFO - Download model 'tiansz/bert-base-chinese' successfully.
C:\Users\17246\AppData\Local\Temp\ipykernel_22948\1373134724.py:24: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return encoding['input_ids'][0],encoding['attention_mask'][

Epoch [1/10],, Loss: 0.0658
Epoch [2/10],, Loss: 0.0539
Epoch [3/10],, Loss: 0.0035
Epoch [4/10],, Loss: 0.0014
Epoch [5/10],, Loss: 0.0019
Epoch [6/10],, Loss: 0.0007
Epoch [7/10],, Loss: 0.0004
Epoch [8/10],, Loss: 0.0013
Epoch [9/10],, Loss: 0.0003
Epoch [10/10],, Loss: 0.0002


In [25]:
model.eval()

text='这个商品真好'
encoding=tokenizer.encode_plus(
    text,
    add_special_tokens=True,
    max_length=128,
    truncation=True,
    padding='max_length',
    return_tensors='pt'
)

input_ids=encoding['input_ids']
attention_mask=encoding['attention_mask']

outputs=model(input_ids,attention_mask)
predicted=torch.argmax(torch.softmax(outputs,dim=1),1)
predicted


tensor([1])